In [21]:
import cv2
import numpy as np
import os
import glob
from pathlib import Path

# ==========================================
# KONFIGURASI FOLDER
# ==========================================
INPUT_DIR = "data valid merah" 
OUTPUT_DIR = "data_cropped" 
CROP_SIZE = 400 

def smart_center_crop_v3(img_path, output_path, crop_size=400):
    # 1. Baca gambar
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"   ❌ Gagal membaca: {img_path}")
        return

    h_img, w_img = img.shape[:2]

    # 2. Ubah ke Grayscale
    if len(img.shape) == 3 and img.shape[2] == 4:
        gray = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2GRAY)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. Threshold 
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    # 4. CARI KONTUR 
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # =======================================================
    # 🌟 INOVASI V3: SAFE ZONE (MENGABAIKAN DINDING/PANTULAN)
    # =======================================================
    valid_contours = []
    
    # Abaikan 15% pinggiran kiri-kanan dan 10% atas-bawah
    margin_x = int(w_img * 0.14) 
    margin_y = int(h_img * 0.10) 

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        cx = x + w // 2
        cy = y + h // 2
        
        # HANYA terima kontur jika posisinya TIDAK menempel di dinding/pinggir
        if (margin_x < cx < w_img - margin_x) and (margin_y < cy < h_img - margin_y):
            valid_contours.append(c)
    # =======================================================

    if not valid_contours:
        # Jika jagung tidak terdeteksi (atau gambar hitam polos), ambil pas tengah
        cx, cy = w_img // 2, h_img // 2
    else:
        # 5. Cari yang terbesar HANYA dari benda yang ada di dalam Safe Zone
        largest_contour = max(valid_contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        cx = x + w // 2
        cy = y + h // 2

    # 6. Hitung batas potongan 400x400
    half = crop_size // 2
    y1 = max(0, cy - half)
    y2 = min(h_img, cy + half)
    x1 = max(0, cx - half)
    x2 = min(w_img, cx + half)

    # Potong gambar
    cropped = img[y1:y2, x1:x2]

    # 7. Padding 
    pad_y1 = max(0, half - cy)
    pad_y2 = max(0, (cy + half) - h_img)
    pad_x1 = max(0, half - cx)
    pad_x2 = max(0, (cx + half) - w_img)

    if pad_y1 > 0 or pad_y2 > 0 or pad_x1 > 0 or pad_x2 > 0:
        if len(img.shape) == 3 and img.shape[2] == 4:
            cropped = cv2.copyMakeBorder(cropped, pad_y1, pad_y2, pad_x1, pad_x2, cv2.BORDER_CONSTANT, value=[0, 0, 0, 0])
        else:
            cropped = cv2.copyMakeBorder(cropped, pad_y1, pad_y2, pad_x1, pad_x2, cv2.BORDER_CONSTANT, value=[0, 0, 0])

    # 8. Simpan hasil
    cv2.imwrite(output_path, cropped)

# ==========================================
# EKSEKUSI MASSAL DENGAN LOG OUTPUT
# ==========================================
print("="*60)
print("🚀 MEMULAI PROSES SMART AUTO-CROPPING V3 (SAFE ZONE ENABLED)")
print("="*60)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

kelas_list = ['1', '2', '3', '4']
total_diproses = 0

for kelas in kelas_list:
    input_kelas_dir = os.path.join(INPUT_DIR, kelas)
    output_kelas_dir = os.path.join(OUTPUT_DIR, kelas)
    
    if not os.path.exists(input_kelas_dir):
        print(f"⚠️ Melewati Kelas {kelas}: Folder tidak ditemukan.")
        continue
        
    Path(output_kelas_dir).mkdir(parents=True, exist_ok=True)
    file_list = glob.glob(os.path.join(input_kelas_dir, "*.png"))
    jumlah_file = len(file_list)
    
    print(f"📂 Memproses Kelas {kelas}: Ditemukan {jumlah_file} file gambar.")
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_kelas_dir, file_name)
        
        smart_center_crop_v3(file_path, output_path, CROP_SIZE)
        total_diproses += 1

print("="*60)
print(f"✅ EKSEKUSI SELESAI! Total {total_diproses} gambar berhasil diproses.")
print(f"📁 Semua hasil disimpan di dalam folder: '{OUTPUT_DIR}'")
print("="*60)

🚀 MEMULAI PROSES SMART AUTO-CROPPING V3 (SAFE ZONE ENABLED)
📂 Memproses Kelas 1: Ditemukan 195 file gambar.
📂 Memproses Kelas 2: Ditemukan 379 file gambar.
📂 Memproses Kelas 3: Ditemukan 239 file gambar.
📂 Memproses Kelas 4: Ditemukan 92 file gambar.
✅ EKSEKUSI SELESAI! Total 905 gambar berhasil diproses.
📁 Semua hasil disimpan di dalam folder: 'data_cropped'
